# 06 — Entanglement & Noise Diagnostics

Reproduce **GAP-5** (entanglement / MI) and **GAP-4** (noise component sensitivity)
figures for the Q-MMF paper.

Sections:
1. Setup
2. Mutual Information across training epochs
3. MI vs prediction correctness
4. MI collapse under depolarizing noise
5. Component-wise noise sensitivity (fusion / decoder-attn / head)
6. Error propagation in autoregressive decoding

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid'); plt.rcParams['figure.dpi'] = 120

## 1 — Setup

In [ ]:
from src.quantum.entanglement import (
    CrossModalEntanglementAnalyzer,
    partial_trace, von_neumann_entropy, mutual_information,
    meyer_wallach_from_rho, make_dm_circuit,
)
from src.quantum.ansatz_metrics import (
    expressibility, entangling_capability, profile_ansatz,
)
from src.quantum.noise_wrapper import (
    apply_component_noise, restore_component_noise,
    ResidualQuantumMitigation, DepolarizingNoiseWrapper,
)
from src.evaluation.noise_analysis import (
    per_step_token_entropy, per_step_disagreement,
    divergence_onset, find_crossover_threshold,
)
print('All imports OK')

## 2 — MI across training epochs

Train a small model and record MI(text:image) each epoch.  
If dataset not available, shows a pre-computed example.

In [ ]:
# Run: python experiments/run_entanglement_diagnostics.py --epochs 8 --device cpu
# Or load saved results if available:
import json
result_path = Path('experiments/results/entanglement_diagnostics.json')
if result_path.exists():
    with open(result_path) as f:
        data = json.load(f)
    mi_epochs = data.get('mi_across_epochs', [])
else:
    print('Run experiments/run_entanglement_diagnostics.py first to generate data')
    mi_epochs = []

In [ ]:
if mi_epochs:
    fig, ax1 = plt.subplots(figsize=(7, 4))
    epochs = [e['epoch'] for e in mi_epochs]
    accs = [e['val_acc'] for e in mi_epochs]
    mis = [e.get('MI_mean') for e in mi_epochs]
    ax1.plot(epochs, accs, 'o-', color='royalblue', label='Val Accuracy')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy', color='royalblue')
    ax2 = ax1.twinx()
    ax2.plot(epochs, mis, 's--', color='crimson', label='MI(text:image)')
    ax2.set_ylabel('Mutual Information (bits)', color='crimson')
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1+lines2, labels1+labels2, loc='lower right')
    plt.title('MI(text:image) vs Validation Accuracy across training')
    plt.tight_layout(); plt.show()

## 3 — MI vs prediction correctness

In [ ]:
if result_path.exists():
    with open(result_path) as f:
        data = json.load(f)
    mc = data.get('mi_vs_correctness', {})
    if mc and mc.get('mi_correct_mean') is not None:
        fig, ax = plt.subplots(figsize=(4, 3.5))
        bars = ax.bar(['Correct', 'Incorrect'],
                       [mc['mi_correct_mean'], mc['mi_incorrect_mean']],
                       color=['seagreen', 'salmon'])
        ax.set_ylabel('Mean MI (bits)')
        ax.set_title(f'MI vs correctness (Δ={mc["delta"]:.3f}, r_pb={mc.get("point_biserial_r", "N/A")})')
        plt.tight_layout(); plt.show()
    else:
        print('mi_vs_correctness data not available')
else:
    print('Load results first')

## 4 — MI collapse under depolarizing noise

In [ ]:
if result_path.exists():
    with open(result_path) as f:
        data = json.load(f)
    mi_noise = data.get('mi_under_noise', [])
    if mi_noise:
        fig, ax = plt.subplots(figsize=(6, 3.5))
        ps = [d['p'] for d in mi_noise]
        mis = [d['MI'] for d in mi_noise]
        ax.plot(ps, mis, 'o-', color='crimson', linewidth=2)
        ax.set_xlabel('Depolarizing noise p'); ax.set_ylabel('MI (bits)')
        ax.set_title('MI collapse under NISQ noise (GAP-5 evidence)')
        plt.tight_layout(); plt.show()

In [ ]:
# Standalone demo with tiny circuit
from math import pi
demo = CrossModalEntanglementAnalyzer(
    n_qubits=6, n_layers=2,
    weights=torch.randn(2, 12) * 0.01)
text_feat = np.random.uniform(-pi/2, pi/2, 3)
image_feat = np.random.uniform(-pi/2, pi/2, 3)
r = demo.analyze_sample(text_feat, image_feat)
print({k: round(v, 4) for k, v in r.items()})

## 5 — Component-wise noise sensitivity

Three PQC positions: **fusion**, **decoder-attention**, **classifier-head**.
GAP-4 evidence: compare degradation curves.

In [ ]:
result_path_ns = Path('experiments/results/noise_study.json')
if result_path_ns.exists():
    with open(result_path_ns) as f:
        ns = json.load(f)
    sweep = ns.get('sweep', [])
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = {'qmmf_fusion': 'royalblue', 'qmmf_quantum_head': 'darkorange',
              'qmmf_hybrid_decoder': 'crimson'}
    labels_map = {'qmmf_fusion': 'Fusion PQC', 'qmmf_quantum_head': 'Head PQC',
                  'qmmf_hybrid_decoder': 'Decoder-Attention PQC'}
    for variant in colors:
        pts = sorted([(r['p'], r['value']) for r in sweep
                       if r['variant'] == variant and r['metric'] == 'accuracy'],
                      key=lambda x: x[0])
        if pts:
            ax.plot([p for p,_ in pts], [v for _,v in pts], 'o-',
                    color=colors[variant], label=labels_map[variant])
    ax.set_xlabel('Noise probability p'); ax.set_ylabel('MSA Accuracy')
    ax.set_title('Component-wise noise sensitivity (GAP-4)')
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print('Run experiments/run_noise_study.py to generate sweep data')

## 6 — Error propagation in autoregressive decoding

In [ ]:
if result_path_ns.exists():
    with open(result_path_ns) as f:
        ns = json.load(f)
    ep = ns.get('error_propagation', {})
    entropy_data = ep.get('entropy', {})
    if entropy_data:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for p_str, ents in entropy_data.items():
            axes[0].plot(ents, label=f'p={p_str}')
        axes[0].set_xlabel('Decode step'); axes[0].set_ylabel('Entropy')
        axes[0].set_title('Per-step token entropy under noise'); axes[0].legend()
        prop = ep.get('propagation', {})
        for p_str, d in prop.items():
            axes[1].plot(d['disagreement_curve'], label=f'p={p_str}')
        axes[1].set_xlabel('Decode step'); axes[1].set_ylabel('Fraction diverged')
        axes[1].set_title('Error propagation (GAP-4)'); axes[1].legend()
        plt.tight_layout(); plt.show()

## 7 — Ansatz profile (GAP-6 / §2.7)

In [ ]:
if result_path.exists():
    with open(result_path) as f:
        data = json.load(f)
    ap = data.get('ansatz_profile', [])
    if ap:
        import pandas as pd
        df = pd.DataFrame([r for r in ap if r.get('entangling_capability') is not None])
        if not df.empty:
            print(df.to_string(index=False))
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            for ansatz in df['ansatz'].unique():
                sub = df[df['ansatz'] == ansatz]
                for q in sub['n_qubits'].unique():
                    d = sub[sub['n_qubits'] == q].sort_values('depth')
                    axes[0].plot(d['depth'], d['expressibility_kl'], 'o-',
                                 label=f'{ansatz} q={q}')
                    axes[1].plot(d['depth'], d['entangling_capability'], 's-',
                                 label=f'{ansatz} q={q}')
            axes[0].set_title('Expressibility (KL vs Haar)'); axes[0].set_xlabel('Depth')
            axes[1].set_title('Entangling Capability (MW)'); axes[1].set_xlabel('Depth')
            for ax in axes: ax.legend(fontsize=7)
            plt.tight_layout(); plt.show()